In [ ]:
# [0] Colab setup — clone repo so src/ is available, install CLIP
import sys, os

REPO = 'https://github.com/sudikshyapant/Sparse-CLIP-with-Spectral-Loss'
REPO_DIR = '/content/Sparse-CLIP-with-Spectral-Loss'

if 'google.colab' in sys.modules:
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO} {REPO_DIR}')
    os.chdir(REPO_DIR)
    os.system('pip install -q git+https://github.com/openai/CLIP.git')

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print('Working dir:', repo_root)

# Variation 1 — InfoNCE vs Spectral Contrastive Loss

In [ ]:
# [1] Config — mounts Drive, sets all paths
from google.colab import drive
drive.mount('/content/drive')

from src.config import CONFIG
print('cache_dir :', CONFIG['cache_dir'])
print('device    :', CONFIG['device'])

In [ ]:
# [2] Download COCO images to local storage (skipped if cache already on Drive)
#
# First run:  downloads images locally → computes embeddings → saves .pt to Drive
# Later runs: cache found on Drive → this cell is a no-op

import os, pathlib

DRIVE_COCO = '/content/drive/MyDrive/sparse_clip/coco'
LOCAL_COCO = '/content/coco'
cache_dir  = CONFIG['cache_dir']

need_train = not (cache_dir / 'train_img_emb.pt').exists()
need_val   = not (cache_dir / 'val_img_emb.pt').exists()

if not need_train and not need_val:
    print('Cache found on Drive — skipping all downloads.')
else:
    os.makedirs(f'{LOCAL_COCO}/annotations', exist_ok=True)

    # Annotations (always needed, tiny)
    for f in ['captions_train2017.json', 'captions_val2017.json']:
        dst = f'{LOCAL_COCO}/annotations/{f}'
        if not os.path.exists(dst):
            os.system(f'cp {DRIVE_COCO}/annotations/{f} {dst}')
            print(f'Copied {f}')

    if need_train:
        print('Downloading train2017 (~18 GB) to local storage...')
        os.system('wget -q http://images.cocodataset.org/zips/train2017.zip -O /tmp/train2017.zip')
        os.system(f'unzip -q /tmp/train2017.zip -d {LOCAL_COCO}')
        os.system('rm /tmp/train2017.zip')
        print('train2017 ready')

    if need_val:
        if os.path.exists(f'{DRIVE_COCO}/val2017'):
            print('Copying val2017 from Drive...')
            os.system(f'cp -r {DRIVE_COCO}/val2017 {LOCAL_COCO}/val2017')
        else:
            print('Downloading val2017 (~1 GB)...')
            os.system('wget -q http://images.cocodataset.org/zips/val2017.zip -O /tmp/val2017.zip')
            os.system(f'unzip -q /tmp/val2017.zip -d {LOCAL_COCO}')
            os.system('rm /tmp/val2017.zip')
        print('val2017 ready')

    # Point CONFIG to local images for fast reading
    CONFIG['coco_train_images'] = pathlib.Path(LOCAL_COCO) / 'train2017'
    CONFIG['coco_val_images']   = pathlib.Path(LOCAL_COCO) / 'val2017'
    CONFIG['coco_train_ann']    = pathlib.Path(LOCAL_COCO) / 'annotations' / 'captions_train2017.json'
    CONFIG['coco_val_ann']      = pathlib.Path(LOCAL_COCO) / 'annotations' / 'captions_val2017.json'
    print('CONFIG paths updated to local storage')

In [ ]:
# [3] Compute and cache CLIP embeddings
# Reads images locally (first run) or loads .pt from Drive (subsequent runs).
import clip, torch
from src.data_utils import cache_or_compute_embeddings, make_loader

device = CONFIG['device']
clip_model, preprocess = clip.load(CONFIG['clip_model'], device=device)
clip_model.eval()

train_img, train_txt = cache_or_compute_embeddings(clip_model, preprocess, 'train', CONFIG)
val_img,   val_txt   = cache_or_compute_embeddings(clip_model, preprocess, 'val',   CONFIG)
print(f'train: {train_img.shape}  val: {val_img.shape}')

train_loader = make_loader(train_img, train_txt, CONFIG['batch_size'])

In [ ]:
# [4] Model factory
from src.model import SparseHead

def make_head():
    return SparseHead(CONFIG['embed_dim'], CONFIG['sparse_dim']).to(device)

In [ ]:
# [5a] Train — InfoNCE with learnable logit scale
import torch.nn as nn, torch.optim as optim
from src.losses import infonce_loss
from src.train  import train_one_epoch, evaluate, save_checkpoint

head_infonce = make_head()
log_scale = nn.Parameter(torch.tensor(CONFIG['log_scale_init'], device=device))

def infonce_fn(img, txt):
    return infonce_loss(img, txt, log_scale)

opt_infonce = optim.AdamW(
    list(head_infonce.parameters()) + [log_scale],
    lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay']
)

loss_curve_infonce = []
for epoch in range(1, CONFIG['epochs'] + 1):
    l = train_one_epoch(head_infonce, train_loader, opt_infonce, infonce_fn, device)
    loss_curve_infonce.append(l)
    print(f'InfoNCE  epoch {epoch:3d}/{CONFIG["epochs"]}  loss={l:.4f}  τ={1/log_scale.exp().item():.4f}')

metrics_infonce = evaluate(head_infonce, val_img, val_txt, CONFIG)
metrics_infonce['loss_curve'] = loss_curve_infonce
print('\nInfoNCE metrics:', metrics_infonce)
save_checkpoint(head_infonce, metrics_infonce, 'infonce', 'variation1', CONFIG)

In [ ]:
# [5b] Train — Spectral contrastive loss
from src.losses import spectral_loss

head_spectral = make_head()
opt_spectral = optim.AdamW(
    head_spectral.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay']
)

loss_curve_spectral = []
for epoch in range(1, CONFIG['epochs'] + 1):
    l = train_one_epoch(head_spectral, train_loader, opt_spectral, spectral_loss, device)
    loss_curve_spectral.append(l)
    print(f'Spectral epoch {epoch:3d}/{CONFIG["epochs"]}  loss={l:.4f}')

metrics_spectral = evaluate(head_spectral, val_img, val_txt, CONFIG)
metrics_spectral['loss_curve'] = loss_curve_spectral
print('\nSpectral metrics:', metrics_spectral)
save_checkpoint(head_spectral, metrics_spectral, 'spectral', 'variation1', CONFIG)

In [ ]:
# [6] Results table
k = CONFIG['retrieval_k']
print(f'\n{"Model":12s}  IR@{k}   TR@{k}   L0_img   L0_txt   Clarity  Cross-modal')
for name, m in [('InfoNCE', metrics_infonce), ('Spectral', metrics_spectral)]:
    print(f"{name:12s}  {m[f'IR@{k}']:.3f}   {m[f'TR@{k}']:.3f}   "
          f"{m['l0_img']:6.1f}   {m['l0_txt']:6.1f}   {m['clarity']:.4f}   {m['cross_modal']:.4f}")

In [ ]:
# [7] Comparison plot
import matplotlib.pyplot as plt

names = ['InfoNCE', 'Spectral']
all_m = [metrics_infonce, metrics_spectral]
k = CONFIG['retrieval_k']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for name, m in zip(names, all_m):
    axes[0].plot(m['loss_curve'], label=name)
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

x = range(len(names))
ir = [m[f'IR@{k}'] for m in all_m]
tr = [m[f'TR@{k}'] for m in all_m]
axes[1].bar([i - 0.2 for i in x], ir, 0.4, label=f'IR@{k}')
axes[1].bar([i + 0.2 for i in x], tr, 0.4, label=f'TR@{k}')
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(names)
axes[1].set_title(f'Retrieval@{k}'); axes[1].legend()

cl  = [m['clarity'] for m in all_m]
l0  = [m['l0_img']  for m in all_m]
ax2 = axes[2].twinx()
axes[2].bar([i - 0.2 for i in x], cl, 0.4, color='steelblue', label='Clarity')
ax2.bar(    [i + 0.2 for i in x], l0, 0.4, color='orange',    label='L0 img')
axes[2].set_xticks(list(x)); axes[2].set_xticklabels(names)
axes[2].set_ylabel('Clarity', color='steelblue')
ax2.set_ylabel('L0', color='orange')
axes[2].set_title('Clarity vs L0')
axes[2].legend(loc='upper left'); ax2.legend(loc='upper right')

fig.tight_layout()
out_path = CONFIG['results_dir'] / 'variation1' / 'comparison.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Saved {out_path}')